# Baseline GPT-2 Math — V3 LoRA r32 Pure Evaluation

Notebook V2 fine-tune `NlpHUST/gpt2-vietnamese` để sinh **lời giải toán tiếng Việt + đáp án cuối**:

```text
Lời giải ...
Đáp án là: <số>
```

**Nâng cấp so với baseline:**

1. **Prompt mới**: có instruction + type hint tiếng Việt → model học format chắc hơn.
2. **Training**: 2 epochs, LR=5e-5, effective batch 32, bf16, `adamw_torch_fused`.
3. **Decoding**: beam search (num_beams=4) + batch inference + custom StoppingCriteria. Bỏ `no_repeat_ngram_size`, bỏ `repetition_penalty`. `max_new_tokens=320`.
4. **Eval fix**: cho phép `allow_last_number=True` cho prediction; auto-append anchor nếu output chỉ có số.
5. **EOS robust**: dùng `tokenizer.eos_token_id` thật, resize model embedding theo `len(tokenizer)` để tránh CUDA index out of bounds.


## Run

Kaggle: GPU ON, Internet OFF. Tổng thời gian dự kiến **<= 3 giờ**.

Output:
- `data/train_preprocessed.json`: train set sau preprocessing và smart truncation.
- `data/train_preprocessing_report.json`: thống kê preprocessing.
- `gpt2_math_baseline_ckpt/`: checkpoint sau fine-tune.
- `valid_output.json` + `valid_report.json`: output và đánh giá chi tiết validation.
- `test_predictions.json`: file nộp cho Phase 2.


In [1]:
# 1. Import và kiểm tra môi trường
import os
import sys
import gc
import re
import json
import math
import time
import random
import hashlib
import inspect
import platform
import shutil
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
    set_seed,
)

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from peft import LoraConfig, get_peft_model, PeftModel
    PEFT_AVAILABLE = True
except Exception as e:
    PEFT_AVAILABLE = False
    LoraConfig = None
    get_peft_model = None
    PeftModel = None
    print("WARNING: peft chưa khả dụng:", repr(e))
    
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 100)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Không bật deterministic mặc định vì có thể làm chậm training trên Kaggle.
# Nếu cần reproduce chặt hơn, có thể bật ở cell config:
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False

print("Python:", sys.version.replace("\n", " "))
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
CUDA_OK = torch.cuda.is_available()
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        capability = torch.cuda.get_device_capability(i)
        print(f"GPU {i}:", name, "| capability:", capability)
    major, minor = torch.cuda.get_device_capability(0)
    if major < 7:
        CUDA_OK = False
        print("WARNING: GPU hiện tại có compute capability < 7.0, không tương thích với PyTorch CUDA hiện tại.")
        print("Hãy chọn GPU T4/V100/A100 thay vì P100 trên Kaggle.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: True | GPU count: 2
GPU 0: Tesla T4 | capability: (7, 5)
GPU 1: Tesla T4 | capability: (7, 5)


In [2]:
# 2. Đường dẫn dữ liệu, model và output
def first_existing(*paths):
    checked = []
    for p in paths:
        p = Path(p)
        checked.append(str(p))
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào:\n" + "\n".join(checked))


def first_existing_optional(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


IS_KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
    PROJECT_ROOT / "data",
    PROJECT_ROOT,
)

MODEL_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/nlphust-gpt2-vietnamese",
    PROJECT_ROOT / "models" / "nlphust-gpt2-vietnamese",
)

WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else PROJECT_ROOT / "outputs" / "baseline_gpt2_math_v2"
WORK_DIR.mkdir(parents=True, exist_ok=True)

GENERATED_DATA_DIR = WORK_DIR / "data" if IS_KAGGLE else PROJECT_ROOT / "data"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"
TEST_FILE = first_existing_optional(DATA_DIR / "test.json", "/kaggle/input/test.json")

OUTPUT_DIR = WORK_DIR / "gpt2_math_baseline_ckpt"
VALID_OUTPUT_PATH = WORK_DIR / "valid_output.json"
VALID_REPORT_PATH = WORK_DIR / "valid_report.json"
TEST_OUTPUT_PATH = WORK_DIR / "test_predictions.json"

SAFE_EOS_ID = 50256
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("WORK_DIR:", WORK_DIR)
print("GENERATED_DATA_DIR:", GENERATED_DATA_DIR)
print("TEST_FILE:", TEST_FILE)


DATA_DIR: /kaggle/input/datasets/kimanh2002/dataset-math
MODEL_DIR: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
WORK_DIR: /kaggle/working
GENERATED_DATA_DIR: /kaggle/working/data
TEST_FILE: None


In [3]:
# 3. Đọc dữ liệu
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
MAX_TEST_SAMPLES = None


def load_records(path, need_response=False):
    path = Path(path)
    with path.open("r", encoding="utf-8-sig") as f:
        first = f.read(1)
        f.seek(0)
        records = json.load(f) if first == "[" else [json.loads(line) for line in f if line.strip()]

    out = []
    for i, rec in enumerate(records):
        if "query_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu query_vi")
        if need_response and "response_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu response_vi")
        item = dict(rec)
        item.setdefault("id", i)
        item.setdefault("type", "unknown")
        out.append(item)
    return out


raw_train = load_records(TRAIN_FILE, need_response=True)
raw_valid = load_records(VALID_FILE, need_response=True) if VALID_FILE.exists() else []
raw_test = load_records(TEST_FILE) if TEST_FILE else []

if MAX_TRAIN_SAMPLES is not None:
    raw_train = raw_train[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES is not None:
    raw_valid = raw_valid[:MAX_VALID_SAMPLES]
if MAX_TEST_SAMPLES is not None:
    raw_test = raw_test[:MAX_TEST_SAMPLES]

print("raw train:", len(raw_train))
print("raw valid:", len(raw_valid))
print("raw test :", len(raw_test))
print(json.dumps(raw_train[0], ensure_ascii=False, indent=2)[:1400])

raw train: 95400
raw valid: 1000
raw test : 0
{
  "original_question_vi": "Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực phẩm luôn chuẩn bị thêm mười đĩa đề phòng trường hợp có sự cố xảy ra. Mỗi đĩa bít tết và măng tây sốt bơ tỏi sẽ có 8 ngọn măng tây trên đó. Người cung cấp thực phẩm sẽ cần tất cả bao nhiêu ngọn măng tây?",
  "original_question_en": "Bridgette and Alex are getting married. Bridgette is inviting 84 guests, and Alex is inviting two thirds of that number of guests. They hired a caterer to make a plated meal for each guest at the wedding reception. The caterer always makes ten extra plates just in case something goes wrong. Each plate of steak and asparagus in garlic butter will have 8 asparagus spears on it. How many asparagus spears will the caterer need in all?",
  "query_vi": "Bridgette và Alex sắp kết hôn. Bridgett

In [4]:
# 4. Hàm trích đáp án và tính điểm
ANSWER_ANCHORS = [
    r"Đáp\s*án\s*là",
    r"Câu\s*trả\s*lời\s*là",
    r"(?:Câu\s+)?Trả\s*lời(?:\s+là)?",
    r"Đáp\s*án(?:\s+[A-Za-z]{1,4}\d{0,2})?",  # bắt cả "Đáp án C4:"
    r"Kết\s*quả\s*là",
    r"Vậy\s*đáp\s*án\s*là",
    r"The answer is",
    r"Answer",
    r"####",
]

ANSWER_ANCHOR_RE = re.compile(
    r"(?:"
    + "|".join(ANSWER_ANCHORS)
    + r")\s*[:：]?",
    flags=re.IGNORECASE,
)

BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
NUM_RE = re.compile(r"[-+]?\d[\d.,]*(?:\s*/\s*[-+]?\d[\d.,]*)?")


def clean_answer_tail(text):
    if text is None:
        return None
    text = str(text).strip().split("\n", 1)[0]
    text = re.sub(r"^(?:là|=|:|：)\s*", "", text, flags=re.IGNORECASE)
    text = text.strip(" .。;；,，]}）)")
    text = text.strip("[{(（")
    return text or None


def first_answer_unit(text):
    """
    Lấy đơn vị đáp án đầu tiên trong một đoạn tail sau anchor.
    Ưu tiên boxed, sau đó số đầu tiên.
    Không lấy số cuối toàn output nữa.
    """
    text = str(text or "")

    box = BOXED_RE.search(text)
    if box:
        return clean_answer_tail(box.group(1))

    num = NUM_RE.search(text)
    if num:
        return clean_answer_tail(num.group(0))

    return None


def last_answer_unit(text):
    """
    Fallback chỉ dùng khi không có anchor.
    """
    text = str(text or "")

    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])

    nums = NUM_RE.findall(text)
    if nums:
        return clean_answer_tail(nums[-1])

    return None


def extract_answer_text(text, allow_last_number=False, prefer_first_anchor=False):
    """
    - Với gold/reference: prefer_first_anchor=False để lấy anchor cuối nếu response có nhiều marker.
    - Với model output: prefer_first_anchor=True để lấy đáp án đầu tiên sau anchor, tránh đuôi rác kiểu
      'Đáp án: 9.5.5.5.8.8...'.
    """
    text = str(text or "")
    matches = list(ANSWER_ANCHOR_RE.finditer(text))

    if matches:
        m = matches[0] if prefer_first_anchor else matches[-1]
        tail = text[m.end():]
        ans = first_answer_unit(tail)
        if ans is not None:
            return ans

    if allow_last_number:
        return last_answer_unit(text)

    return None


def parse_plain_number(text):
    text = str(text).strip().replace(" ", "")
    if not text:
        return None

    if "/" in text:
        parts = text.split("/")
        if len(parts) == 2:
            a = parse_plain_number(parts[0])
            b = parse_plain_number(parts[1])
            if a is not None and b not in (None, 0):
                return a / b
        return None

    if re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", text):
        text = text.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", text):
        text = text.replace(",", "")
    elif "," in text and "." not in text:
        right = text.split(",")[-1]
        text = text.replace(",", "") if len(right) == 3 else text.replace(",", ".")
    elif "," in text and "." in text:
        text = text.replace(",", "")

    try:
        out = float(text)
    except ValueError:
        return None
    return out if math.isfinite(out) else None


def parse_number(text):
    if text is None:
        return None
    text = str(text).strip()
    if not text:
        return None
    direct = parse_plain_number(text)
    if direct is not None:
        return direct
    m = NUM_RE.search(text)
    return parse_plain_number(m.group(0)) if m else None


def relative_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))


def score_one(rel_err, extractable=True):
    if not extractable or rel_err is None:
        return 0
    if rel_err <= 0.01:
        return 10
    if rel_err <= 0.10:
        return 5
    if rel_err <= 0.50:
        return 1
    return 0

In [5]:
# 5. Data processing trước khi train (Bước 1, 2, 3 + dedup nhẹ)
DROP_TRAIN_WITHOUT_FINAL_ANSWER = True
SAVE_PREPROCESSED_TRAIN = True
DEDUP_TRAIN = True   # V2: bật dedup nhẹ để giảm noise
PREPROCESSED_TRAIN_FILE = GENERATED_DATA_DIR / "train_preprocessed.json"
PREPROCESSING_REPORT_FILE = GENERATED_DATA_DIR / "train_preprocessing_report.json"


def normalize_space(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def word_count(text):
    return len(re.findall(r"\S+", str(text or "")))


def normalized_hash(text):
    text = normalize_space(text).lower()
    return hashlib.blake2b(text.encode("utf-8"), digest_size=16).hexdigest()


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_records_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def load_jsonl_records(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def dataset_fingerprint(records):
    content = json.dumps(
        [r.get("query_vi", "") + "\n" + r.get("response_vi", "") for r in records],
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.md5(content).hexdigest()


def fix_artifacts(text):
    text = str(text or "")
    text = text.replace(r"\đóng hộp{", r"\boxed{")
    text = text.replace("\u200b", "").replace("\ufeff", "")
    return text.strip()


def strip_asy_blocks(text):
    text = str(text or "")
    text = re.sub(r"\[asy\].*?\[/asy\]", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(
        r"\[asy\].*?(?=(?:Giá trị của|Giá trị là|Câu trả lời|Đáp án|Nếu chúng ta biết|Để giải|$))",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    text = re.sub(r"\[/asy\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def clean_text(text):
    text = str(text or "")
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(Giá trị của biến [^\n?]+\?)\s*\1", r"\1", text)
    if re.search(r"Đáp án là|Câu trả lời là|####|\\boxed", text, flags=re.IGNORECASE):
        text = re.sub(
            r"\n(?:The answer is[:\s]+[\d.,/\\{}a-zA-Z]+\.?\s*)+$",
            "",
            text,
            flags=re.IGNORECASE,
        )
    return text.strip()


def normalize_decimal_format(text):
    text = str(text or "")
    text = re.sub(
        r"(?<![{\\])(\d+)\.(\d{3}),(\d{1,3})(?!\d)",
        lambda m: f"{m.group(1)}{m.group(2)}.{m.group(3)}",
        text,
    )
    text = re.sub(
        r"(?<![a-zA-ZÀ-ỹ{])(-?0),(\d{1,3})(?!\d)(?!})",
        lambda m: f"{m.group(1)}.{m.group(2)}",
        text,
    )
    text = re.sub(
        r"(?<![a-zA-ZÀ-ỹ{])(-?\d+),(\d{1,2})(?!\d)(?!})",
        lambda m: f"{m.group(1)}.{m.group(2)}",
        text,
    )
    return text


def preprocess_step2(query, response):
    query = strip_asy_blocks(query)
    response = strip_asy_blocks(response)
    query = clean_text(query)
    response = clean_text(response)
    query = normalize_decimal_format(query)
    response = normalize_decimal_format(response)
    return query, response


def extract_final_answer(response):
    text = str(response or "")
    anchor_re = re.compile(
        r"(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer|####)\s*[:：]?",
        flags=re.IGNORECASE,
    )
    matches = list(anchor_re.finditer(text))
    if matches:
        return clean_answer_tail(text[matches[-1].end():])

    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])

    numbers = re.findall(
        r"(?:\\frac\{[^}]+\}\{[^}]+\}|[-+]?\d+(?:[.,]\d+)?(?:\s*\\[a-zA-Z]+\{[^}]*\})*)",
        text,
    )
    if numbers:
        return clean_answer_tail(numbers[-1])
    return None


def normalize_answer(answer):
    answer = clean_answer_tail(answer) or ""
    answer = re.sub(r"\s+", " ", answer).strip()
    answer = re.sub(r"\(([-+]?\d+),([-+]?\d+)\)", r"(\1, \2)", answer)

    if not re.search(r"[\\{^_]", answer):
        if re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", answer) and not re.fullmatch(r"[-+]?0,\d{3}", answer):
            answer = answer.replace(",", "")
        elif re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", answer):
            answer = answer.replace(".", "").replace(",", ".")
        else:
            answer = normalize_decimal_format(answer)
        answer = re.sub(r"^([-+]?\d[\d./]*)(?:\s+[a-zA-ZÀ-ỹ%].*)$", r"\1", answer)
    else:
        answer = normalize_decimal_format(answer)

    return answer.strip(" .。;；,，")


def rebuild_response(response, answer):
    cleaned = str(response or "").strip()
    cleaned = re.sub(
        r"\s*(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer)\s*[:：]?\s*[^\n]*\s*$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\s*####\s*[^\n]*\s*$", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\n?\s*\\boxed\s*\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}\s*[.。]?\s*$", "", cleaned)
    cleaned = cleaned.rstrip()
    return (cleaned + f"\nĐáp án là: {answer}").strip()


def preprocess_labeled_record(rec, idx, drop_without_answer):
    query_raw = fix_artifacts(rec.get("query_vi"))
    response_raw = fix_artifacts(rec.get("response_vi"))
    if not query_raw or not response_raw:
        return None, "missing_query_or_response"

    query, response = preprocess_step2(query_raw, response_raw)
    answer = normalize_answer(extract_final_answer(response))
    if not answer and drop_without_answer:
        return None, "extract_failed"

    if answer:
        response = rebuild_response(response, answer)

    item = {
        "id": rec.get("id", idx),
        "query_vi": query,
        "response_vi": response,
        "type": rec.get("type", "unknown"),
        "answer_text": answer or None,
        "answer_num": parse_number(answer) if answer else None,
    }
    return item, None


def process_train(records):
    kept = []
    drop_reasons = []
    failed_samples = []
    asy_stripped = 0

    for i, rec in enumerate(tqdm(records, desc="text preprocessing")):
        raw_joined = f"{rec.get('query_vi', '')}\n{rec.get('response_vi', '')}".lower()
        had_asy = "[asy]" in raw_joined
        item, reason = preprocess_labeled_record(rec, i, DROP_TRAIN_WITHOUT_FINAL_ANSWER)
        if reason:
            drop_reasons.append(reason)
            if reason == "extract_failed" and len(failed_samples) < 50:
                failed_samples.append({
                    "index": i,
                    "type": rec.get("type", "unknown"),
                    "query_vi": normalize_space(rec.get("query_vi"))[:180],
                    "response_tail": str(rec.get("response_vi", ""))[-300:],
                })
            continue
        if had_asy:
            asy_stripped += 1
        kept.append(item)

    return kept, Counter(drop_reasons), {
        "extract_failed_preview": failed_samples,
        "asy_stripped_count": asy_stripped,
    }


DEDUP_KEY_MODE = "query_response"  # options: query_response, query_answer, original_response


def count_by_type(records):
    return dict(Counter(r.get("type", "unknown") for r in records))


def make_dedup_key(rec):
    q = normalize_space(rec.get("query_vi", "")).lower()
    answer = str(rec.get("answer_text", "")).strip()
    response = normalize_space(rec.get("response_vi", "")).lower()

    original_hash = rec.get("original_question_hash")
    response_hash = normalized_hash(response)

    if DEDUP_KEY_MODE == "query_response":
        # Khuyến nghị V3: giữ đa dạng lời giải, không drop mạnh các bài cùng đáp án.
        return (q, response)

    if DEDUP_KEY_MODE == "original_response" and original_hash:
        return (str(original_hash), response_hash)

    # fallback V2 cũ
    return (q, answer)


def dedup_records(records):
    """
    V3: dedup theo (query_norm, response_norm), không còn theo (query_norm, answer_text).
    Mục tiêu: tránh drop quá mạnh AnsAug chỉ vì cùng đáp án.
    """
    seen = set()
    out = []
    dup = 0
    dropped_by_type = Counter()

    before_by_type = Counter(r.get("type", "unknown") for r in records)

    for rec in records:
        key = make_dedup_key(rec)
        rec_type = rec.get("type", "unknown")

        if key in seen:
            dup += 1
            dropped_by_type[rec_type] += 1
            continue

        seen.add(key)
        out.append(rec)

    after_by_type = Counter(r.get("type", "unknown") for r in out)

    report = {
        "dedup_key_mode": DEDUP_KEY_MODE,
        "before_by_type": dict(before_by_type),
        "after_by_type": dict(after_by_type),
        "dropped_by_type": dict(dropped_by_type),
        "kept_ratio_by_type": {
            t: round(after_by_type.get(t, 0) / max(1, before_by_type.get(t, 0)), 4)
            for t in sorted(before_by_type)
        },
    }

    return out, dup, report


def process_eval_or_test(records, has_response):
    out = []
    for i, rec in enumerate(records):
        query_raw = fix_artifacts(rec.get("query_vi"))
        query = normalize_decimal_format(clean_text(strip_asy_blocks(query_raw)))
        item = {
            "id": rec.get("id", i),
            "query_vi": query,
            "type": rec.get("type", "unknown"),
        }
        if has_response:
            response_raw = fix_artifacts(rec.get("response_vi"))
            _, response = preprocess_step2(query_raw, response_raw)
            answer = normalize_answer(extract_final_answer(response))
            if answer:
                response = rebuild_response(response, answer)
            item["response_vi"] = response
            item["answer_text"] = answer or None
            item["answer_num"] = parse_number(answer) if answer else None
        out.append(item)
    return out


train_records, drop_counter, preprocess_logs = process_train(raw_train)
n_before_dedup = len(train_records)

if DEDUP_TRAIN:
    train_records, n_dup, dedup_report = dedup_records(train_records)
    print(f"Dedup mode={DEDUP_KEY_MODE}: bỏ {n_dup} mẫu trùng. Còn {len(train_records)} mẫu.")
    print("Dedup kept_ratio_by_type:")
    print(json.dumps(dedup_report["kept_ratio_by_type"], ensure_ascii=False, indent=2))
else:
    n_dup = 0
    dedup_report = {
        "dedup_key_mode": None,
        "before_by_type": count_by_type(train_records),
        "after_by_type": count_by_type(train_records),
        "dropped_by_type": {},
        "kept_ratio_by_type": {},
    }
    

valid_records = process_eval_or_test(raw_valid, has_response=True)
test_records = process_eval_or_test(raw_test, has_response=False)

preprocessing_report = {
    "train_before": len(raw_train),
    "train_after_text_preprocessing": n_before_dedup,
    "train_after_dedup": len(train_records),
    "dropped_text_preprocessing": sum(drop_counter.values()),
    "drop_reasons_text_preprocessing": dict(drop_counter),
    "n_duplicates_removed": n_dup,
    "valid": len(valid_records),
    "test": len(test_records),
    "fingerprint_text_preprocessing": dataset_fingerprint(train_records),
    "dedup_report": dedup_report,
    "dedup_key_mode": DEDUP_KEY_MODE,
    **preprocess_logs,
}

print("train before:", len(raw_train), "| after text preprocessing:", n_before_dedup, "| dropped:", sum(drop_counter.values()))
print("drop reasons:", dict(drop_counter))
print("[asy] stripped in train:", preprocess_logs["asy_stripped_count"])
print("valid:", len(valid_records), "| test:", len(test_records))
print("File train mới sẽ được ghi sau smart truncation:", PREPROCESSED_TRAIN_FILE)
print("\nTarget sau xử lý:")
print(train_records[0]["response_vi"][:800])


text preprocessing:   0%|          | 0/95400 [00:00<?, ?it/s]

Dedup mode=query_response: bỏ 126 mẫu trùng. Còn 95268 mẫu.
Dedup kept_ratio_by_type:
{
  "GSM_AnsAug": 0.9983,
  "GSM_FOBAR": 0.9993,
  "GSM_Rephrased": 1.0,
  "GSM_SV": 0.9953,
  "MATH_AnsAug": 0.9983,
  "MATH_FOBAR": 0.9995,
  "MATH_Rephrased": 0.9995,
  "MATH_SV": 0.9989
}
train before: 95400 | after text preprocessing: 95394 | dropped: 6
drop reasons: {'extract_failed': 6}
[asy] stripped in train: 953
valid: 1000 | test: 0
File train mới sẽ được ghi sau smart truncation: /kaggle/working/data/train_preprocessed.json

Target sau xử lý:
Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó, tức là 84 * 2/3 = 56 khách. Vậy tổng số khách là 84 + 56 = 140 khách. Người phục vụ luôn làm thêm 10 đĩa nên tổng số đĩa cần dùng là 140 + 10 = 150 đĩa. Mỗi đĩa sẽ có 8 ngọn măng tây trên đó, như vậy tổng số ngọn măng tây cần thiết là 150 * 8 = 1200 ngọn măng tây.
Đáp án là: 1200


In [6]:
# 6. Kiểm tra dữ liệu sau processing
def feature_df(records, split):
    rows = []
    for i, rec in enumerate(records):
        rows.append({
            "split": split,
            "index": i,
            "type": rec.get("type", "unknown"),
            "query_words": word_count(rec.get("query_vi")),
            "response_words": word_count(rec.get("response_vi", "")),
            "has_answer": rec.get("answer_text") is not None,
            "answer_num": rec.get("answer_num"),
            "ends_with_anchor": str(rec.get("response_vi", "")).rstrip().endswith("Đáp án là: " + str(rec.get("answer_text", ""))),
        })
    return pd.DataFrame(rows)


train_df = feature_df(train_records, "train")
valid_df = feature_df(valid_records, "valid") if valid_records else pd.DataFrame()

print("Phân bố type sau processing:")
display(train_df["type"].value_counts().rename_axis("type").reset_index(name="count"))

print("Độ dài train (words):")
display(train_df[["query_words", "response_words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))

print("Độ dài và answer rate theo type:")
by_type = (
    train_df.groupby("type")
    .agg(
        count=("index", "count"),
        query_p95=("query_words", lambda s: s.quantile(0.95)),
        response_p95=("response_words", lambda s: s.quantile(0.95)),
        answer_rate=("has_answer", "mean"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)
display(by_type.round(3))

print("Tỷ lệ có final_answer:", round(float(train_df["has_answer"].mean()), 4))
print("Anchor không nằm cuối response:", int((~train_df["ends_with_anchor"]).sum()) if len(train_df) else 0)

Phân bố type sau processing:


,type,count
0,GSM_Rephrased,20028
1,GSM_AnsAug,18713
2,MATH_AnsAug,16967
3,MATH_Rephrased,12468
4,GSM_FOBAR,10016
5,GSM_SV,9823
6,MATH_FOBAR,3666
7,MATH_SV,3587


Độ dài train (words):


,query_words,response_words
count,95268.00,95268.00
mean,48.35,112.98
std,25.89,65.81
min,2.00,4.00
50%,46.00,97.00
90%,82.00,199.00
95%,94.00,236.00
99%,122.00,334.00
max,311.00,771.00


Độ dài và answer rate theo type:


,type,count,query_p95,response_p95,answer_rate
2,GSM_Rephrased,20028,80.00,154.00,1.0
0,GSM_AnsAug,18713,92.00,152.00,1.0
4,MATH_AnsAug,16967,62.00,171.70,1.0
6,MATH_Rephrased,12468,58.00,182.00,1.0
1,GSM_FOBAR,10016,119.25,234.00,1.0
3,GSM_SV,9823,103.00,278.00,1.0
5,MATH_FOBAR,3666,110.00,427.75,1.0
7,MATH_SV,3587,98.00,352.00,1.0


Tỷ lệ có final_answer: 1.0
Anchor không nằm cuối response: 0


In [7]:
# 7. Prompt V2, tokenizer, smart truncation, token audit
MAX_LENGTH = 512
HARD_TRUNC_TYPES = {"MATH_SV", "MATH_FOBAR"}
TRUNCATION_MARKER = "\n...\n"
TOKEN_AUDIT_SAMPLES = 2000

# V3 PROMPT: explicit task-conditioning theo cấu trúc target
TASK_GROUP_MAP = {
    "GSM_AnsAug": "DIRECT_ANSWER",
    "GSM_Rephrased": "DIRECT_ANSWER",
    "MATH_AnsAug": "DIRECT_ANSWER",
    "MATH_Rephrased": "DIRECT_ANSWER",
    "GSM_SV": "SOLVE_FOR_VARIABLE",
    "MATH_SV": "SOLVE_FOR_VARIABLE",
    "GSM_FOBAR": "REVERSE_PARAM",
    "MATH_FOBAR": "REVERSE_PARAM",
    "unknown": "DIRECT_ANSWER",
}

TYPE_LABEL_MAP = {
    "GSM_AnsAug":     "Bài toán số học đời sống - hỏi đáp án trực tiếp",
    "GSM_Rephrased":  "Bài toán số học đời sống - hỏi đáp án trực tiếp",
    "MATH_AnsAug":    "Bài toán nâng cao - hỏi đáp án trực tiếp",
    "MATH_Rephrased": "Bài toán nâng cao - hỏi đáp án trực tiếp",
    "GSM_SV":         "Bài toán số học đời sống - tìm biến chưa biết",
    "MATH_SV":        "Bài toán nâng cao - tìm biến chưa biết",
    "GSM_FOBAR":      "Bài toán số học đời sống - suy ngược tham số",
    "MATH_FOBAR":     "Bài toán nâng cao - suy ngược tham số",
    "unknown":        "Bài toán",
}

TASK_INSTRUCTION_MAP = {
    "DIRECT_ANSWER": "Tìm đáp án cuối cùng đúng với câu hỏi trong đề.",
    "SOLVE_FOR_VARIABLE": "Tìm giá trị của biến/chưa biết được hỏi, không trả lời lại đáp án gốc nếu đề đã biến đổi mục tiêu.",
    "REVERSE_PARAM": "Suy ngược tham số hoặc dữ kiện cần thiếu sao cho điều kiện trong đề đúng.",
}

INSTRUCTION = 'Giải bài toán sau từng bước. Kết thúc bằng dòng "Đáp án là: <số>".'

PROMPT_TEMPLATE = (
    "{instr}\n\n"
    "[TASK:{task_group}]\n"
    "[Loại: {type_label}]\n"
    "Mục tiêu: {task_instruction}\n"
    "Bài toán: {q}\n\n"
    "Lời giải:\n"
)


def get_task_group(t):
    return TASK_GROUP_MAP.get(t, TASK_GROUP_MAP["unknown"])


def get_type_label(t):
    return TYPE_LABEL_MAP.get(t, TYPE_LABEL_MAP["unknown"])


def get_task_instruction(t):
    return TASK_INSTRUCTION_MAP[get_task_group(t)]


def build_prompt(rec):
    rec_type = rec.get("type", "unknown")
    return PROMPT_TEMPLATE.format(
        instr=INSTRUCTION,
        task_group=get_task_group(rec_type),
        type_label=get_type_label(rec_type),
        task_instruction=get_task_instruction(rec_type),
        q=str(rec.get("query_vi", "")).strip(),
    )


tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)

# Verify EOS thật của tokenizer; fallback về SAFE_EOS_ID nếu None.
# Lưu ý: checkpoint GPT-2 có thể có embedding nhỏ hơn len(tokenizer),
# nên model sẽ được resize ở cell train/inference trước khi dùng PAD/EOS này.
real_eos = tokenizer.eos_token_id
EOS_ID = int(real_eos) if real_eos is not None else SAFE_EOS_ID
PAD_ID = EOS_ID
tokenizer.pad_token_id = PAD_ID
tokenizer.eos_token_id = EOS_ID
if getattr(tokenizer, "pad_token", None) is None and getattr(tokenizer, "eos_token", None) is not None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocab_size:", getattr(tokenizer, "vocab_size", None), "| len:", len(tokenizer))
print(f"EOS_ID (used): {EOS_ID} | PAD_ID: {PAD_ID} | tokenizer.eos_token_id thật: {real_eos}")


def encode_no_special(text):
    return tokenizer(str(text or ""), add_special_tokens=False)["input_ids"]


def split_response_for_truncation(response, answer):
    response = str(response or "").rstrip()
    m = re.search(r"\nĐáp án là:\s*([^\n]+)\s*$", response, flags=re.IGNORECASE)
    if m:
        return response[:m.start()].rstrip(), "\nĐáp án là: " + m.group(1).strip()
    suffix = "\nĐáp án là: " + str(answer or extract_answer_text(response, allow_last_number=True) or "").strip()
    body = re.sub(r"\n?Đáp án là:\s*[^\n]+\s*$", "", response, flags=re.IGNORECASE).rstrip()
    return body, suffix


def measure_record_tokens(rec):
    prompt_ids = encode_no_special(build_prompt(rec))
    response_ids = encode_no_special(rec.get("response_vi", "")) + [EOS_ID]
    return len(prompt_ids), len(response_ids), len(prompt_ids) + len(response_ids)


def decode_ids(ids):
    return tokenizer.decode(ids, skip_special_tokens=True).strip()


def make_truncated_body(body_ids, middle_budget, rec_type):
    """
    V3 truncation:
    - Không chỉ giữ tail như V2.
    - Với MATH_SV/MATH_FOBAR: giữ nhiều phần đầu hơn vì phần đầu thường chứa setup phương trình.
    - Vẫn giữ tail để không mất bước kết luận trước anchor.
    """
    if len(body_ids) <= middle_budget:
        return decode_ids(body_ids)

    if middle_budget <= 32:
        # Quá ít budget: fallback giữ tail như cũ.
        return decode_ids(body_ids[-middle_budget:])

    marker_ids = encode_no_special(TRUNCATION_MARKER)
    marker_budget = len(marker_ids)

    available = max(1, middle_budget - marker_budget)

    if rec_type in HARD_TRUNC_TYPES:
        head_budget = int(available * 0.70)
        tail_budget = available - head_budget
    else:
        head_budget = int(available * 0.45)
        tail_budget = available - head_budget

    head_budget = max(1, head_budget)
    tail_budget = max(1, tail_budget)

    head_text = decode_ids(body_ids[:head_budget])
    tail_text = decode_ids(body_ids[-tail_budget:])

    return (head_text + TRUNCATION_MARKER + tail_text).strip()


def smart_truncate_record(rec, max_length):
    prompt_ids = encode_no_special(build_prompt(rec))
    response_ids = encode_no_special(rec.get("response_vi", "")) + [EOS_ID]
    original_length = len(prompt_ids) + len(response_ids)

    item = dict(rec)
    item["original_length"] = original_length
    item["was_truncated"] = False
    item["truncation_strategy"] = "none"

    if original_length <= max_length:
        item["prompt_tokens"] = len(prompt_ids)
        item["response_tokens"] = len(response_ids)
        item["total_tokens"] = original_length
        return item, None

    body, suffix = split_response_for_truncation(rec.get("response_vi", ""), rec.get("answer_text"))
    suffix_ids = encode_no_special(suffix) + [EOS_ID]
    middle_budget = max_length - len(prompt_ids) - len(suffix_ids)

    if middle_budget <= 0:
        return None, "too_long_prompt_or_answer"

    body_ids = encode_no_special(body)
    rec_type = rec.get("type", "unknown")

    while True:
        body_text = make_truncated_body(body_ids, middle_budget, rec_type)
        new_response = (body_text.rstrip() + suffix) if body_text else suffix.lstrip()
        new_response_ids = encode_no_special(new_response) + [EOS_ID]
        new_total = len(prompt_ids) + len(new_response_ids)

        if new_total <= max_length:
            item["response_vi"] = new_response
            item["was_truncated"] = True
            item["truncation_strategy"] = "head_tail_for_hard_type" if rec_type in HARD_TRUNC_TYPES else "head_tail"
            item["prompt_tokens"] = len(prompt_ids)
            item["response_tokens"] = len(new_response_ids)
            item["total_tokens"] = new_total
            return item, None

        overflow = new_total - max_length
        middle_budget -= max(1, overflow)

        if middle_budget <= 0:
            return None, "too_long_after_truncation"


def apply_token_length_policy(records, max_length):
    kept = []
    counter = Counter()
    examples = []
    for rec in tqdm(records, desc="smart truncation"):
        item, reason = smart_truncate_record(rec, max_length)
        if reason:
            counter[reason] += 1
            if len(examples) < 20:
                examples.append({
                    "id": rec.get("id"),
                    "type": rec.get("type"),
                    "reason": reason,
                    "query_vi": rec.get("query_vi", "")[:180],
                })
            continue
        if item.get("was_truncated"):
            counter["smart_truncated"] += 1
        kept.append(item)
    return kept, counter, examples


train_records, token_policy_counter, token_policy_examples = apply_token_length_policy(train_records, MAX_LENGTH)
preprocessing_report.update({
    "max_length": MAX_LENGTH,
    "train_after_token_policy": len(train_records),
    "dropped_token_policy": int(token_policy_counter.get("too_long_prompt_or_answer", 0) + token_policy_counter.get("too_long_after_truncation", 0)),
    "smart_truncated": int(token_policy_counter.get("smart_truncated", 0)),
    "token_policy_counter": dict(token_policy_counter),
    "token_policy_drop_preview": token_policy_examples,
    "fingerprint_final": dataset_fingerprint(train_records),
    "prompt_template_v3": PROMPT_TEMPLATE,
    "instruction": INSTRUCTION,
    "type_label_map": TYPE_LABEL_MAP,
    "task_group_map": TASK_GROUP_MAP,
    "task_instruction_map": TASK_INSTRUCTION_MAP,
})

if SAVE_PREPROCESSED_TRAIN:
    save_records_jsonl(train_records, PREPROCESSED_TRAIN_FILE)
    save_json(preprocessing_report, PREPROCESSING_REPORT_FILE)
    print("Wrote:", PREPROCESSED_TRAIN_FILE)
    print("Wrote:", PREPROCESSING_REPORT_FILE)

    train_records = load_jsonl_records(PREPROCESSED_TRAIN_FILE)
    TRAIN_SOURCE = f"preprocessed_file:{PREPROCESSED_TRAIN_FILE}"
else:
    TRAIN_SOURCE = "preprocessed_in_memory"

print("TRAIN_SOURCE:", TRAIN_SOURCE)
print("Train records used by Trainer:", len(train_records))
assert train_records, "Không có mẫu train sau preprocessing"
assert max(r.get("total_tokens", 0) for r in train_records) <= MAX_LENGTH, "Còn mẫu vượt MAX_LENGTH"

sample = train_records if len(train_records) <= TOKEN_AUDIT_SAMPLES else random.sample(train_records, TOKEN_AUDIT_SAMPLES)
token_rows = []
for rec in tqdm(sample, desc="token audit"):
    p_tokens, r_tokens, total_tokens = measure_record_tokens(rec)
    token_rows.append({
        "type": rec.get("type"),
        "prompt_tokens": p_tokens,
        "response_tokens": r_tokens,
        "total_tokens": total_tokens,
        "will_truncate": total_tokens > MAX_LENGTH,
        "was_truncated": bool(rec.get("was_truncated")),
    })

token_df = pd.DataFrame(token_rows)

token_by_type = (
    token_df.groupby("type")
    .agg(
        n=("type", "size"),
        avg_total_tokens=("total_tokens", "mean"),
        p95_total_tokens=("total_tokens", lambda x: float(x.quantile(0.95))),
        truncated_rate=("was_truncated", "mean"),
    )
    .reset_index()
)

display(token_by_type.sort_values("truncated_rate", ascending=False))

preprocessing_report["token_policy_by_type"] = token_by_type.to_dict("records")
preprocessing_report["hard_trunc_types"] = sorted(HARD_TRUNC_TYPES)
preprocessing_report["truncation_marker"] = TRUNCATION_MARKER
save_json(preprocessing_report, PREPROCESSING_REPORT_FILE)
print("Updated preprocessing report with token_policy_by_type:", PREPROCESSING_REPORT_FILE)

display(token_df[["prompt_tokens", "response_tokens", "total_tokens"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))
print("Tỷ lệ còn vượt MAX_LENGTH:", round(float(token_df["will_truncate"].mean()), 4))
print("Số mẫu smart truncated:", int(token_policy_counter.get("smart_truncated", 0)))
print("Số mẫu drop vì quá dài:", preprocessing_report["dropped_token_policy"])

print("\n--- Sample prompt V2 ---")
print(build_prompt(train_records[0]))
print("--- end ---")


Tokenizer vocab_size: 50257 | len: 50258
EOS_ID (used): 50257 | PAD_ID: 50257 | tokenizer.eos_token_id thật: 50257


smart truncation:   0%|          | 0/95268 [00:00<?, ?it/s]

Wrote: /kaggle/working/data/train_preprocessed.json
Wrote: /kaggle/working/data/train_preprocessing_report.json
TRAIN_SOURCE: preprocessed_file:/kaggle/working/data/train_preprocessed.json
Train records used by Trainer: 95257


token audit:   0%|          | 0/2000 [00:00<?, ?it/s]

,type,n,avg_total_tokens,p95_total_tokens,truncated_rate
5,MATH_FOBAR,74,434.094595,512.0,0.405405
7,MATH_SV,72,377.944444,512.0,0.208333
4,MATH_AnsAug,331,273.066465,512.0,0.054381
3,GSM_SV,201,392.298507,512.0,0.049751
1,GSM_FOBAR,207,348.768116,498.1,0.048309
6,MATH_Rephrased,271,267.169742,439.5,0.029520
0,GSM_AnsAug,415,241.932530,347.5,0.002410
2,GSM_Rephrased,429,235.827506,335.6,0.000000


Updated preprocessing report with token_policy_by_type: /kaggle/working/data/train_preprocessing_report.json


,prompt_tokens,response_tokens,total_tokens
count,2000.00,2000.00,2000.00
mean,137.63,149.74,287.37
std,31.86,77.06,94.87
min,84.00,9.00,130.00
50%,133.00,132.00,265.00
90%,176.00,266.00,433.10
95%,192.00,309.05,504.00
99%,238.00,365.00,512.00
max,355.00,421.00,512.00


Tỷ lệ còn vượt MAX_LENGTH: 0.0
Số mẫu smart truncated: 4220
Số mẫu drop vì quá dài: 11

--- Sample prompt V2 ---
Giải bài toán sau từng bước. Kết thúc bằng dòng "Đáp án là: <số>".

[TASK:DIRECT_ANSWER]
[Loại: Bài toán số học đời sống - hỏi đáp án trực tiếp]
Mục tiêu: Tìm đáp án cuối cùng đúng với câu hỏi trong đề.
Bài toán: Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực phẩm luôn chuẩn bị thêm mười đĩa đề phòng trường hợp có sự cố xảy ra. Mỗi đĩa bít tết và măng tây sốt bơ tỏi sẽ có 8 ngọn măng tây trên đó. Người cung cấp thực phẩm sẽ cần tất cả bao nhiêu ngọn măng tây?

Lời giải:

--- end ---


In [8]:
# 8. Dataset cho supervised fine-tuning (loss mask trên prompt)
def clamp_ids(ids, vocab_size):
    return [min(max(int(x), 0), vocab_size - 1) for x in ids]


def fit_prompt_response(prompt_ids, response_ids, max_length):
    if len(prompt_ids) + len(response_ids) <= max_length:
        return prompt_ids, response_ids
    room = max_length - len(prompt_ids)
    if room <= 0:
        prompt_ids = prompt_ids[: max_length - 1]
        room = max_length - len(prompt_ids)
    response_ids = response_ids[-room:] if room > 0 else []
    return prompt_ids, response_ids


class MathDataset(Dataset):
    def __init__(self, records, tokenizer, vocab_size, max_length):
        self.records = records
        self.tokenizer = tokenizer
        self.vocab_size = vocab_size
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        prompt_ids = self.tokenizer(build_prompt(rec), add_special_tokens=False)["input_ids"]
        response_ids = self.tokenizer(rec["response_vi"], add_special_tokens=False)["input_ids"] + [EOS_ID]
        prompt_ids, response_ids = fit_prompt_response(prompt_ids, response_ids, self.max_length)

        input_ids = clamp_ids(prompt_ids + response_ids, self.vocab_size)
        labels = [-100] * len(prompt_ids) + clamp_ids(response_ids, self.vocab_size)
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }


@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        max_len = int(math.ceil(max_len / 8) * 8)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for item in batch:
            pad = max_len - len(item["input_ids"])
            out["input_ids"].append(item["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(item["attention_mask"] + [0] * pad)
            out["labels"].append(item["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


In [9]:
# 9. Tham số train V3 — reasoning target + task-conditioning + LoRA r32
RUN_TRAIN = True
EXPERIMENT_NAME = "v4_clean_data_fast"

EPOCHS = 2                       # giữ 2 nếu vẫn dùng data cũ; dùng 3 khi đã có clean data nhỏ hơn
PER_DEVICE_BATCH_SIZE = 8        # nếu OOM thì hạ lại 4
GRAD_ACCUM = 4                   # effective batch = 64 nếu có 2 GPU, =32 nếu 1 GPU
LEARNING_RATE = 7e-5             # theo plan khi tăng effective batch
WARMUP_RATIO = 0.02
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 50

TRAINER_EVAL_SAMPLES = 500

# V3 LoRA capacity experiment
USE_LORA = True
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["c_attn", "c_proj", "c_fc"]


def ensure_model_token_embeddings(model, tok, pad_id, eos_id):
    """Resize embedding nếu tokenizer có token id ngoài vocab của checkpoint."""
    token_count = len(tok)
    embed_count = model.get_input_embeddings().num_embeddings
    if token_count > embed_count:
        print(f"Resize token embeddings: {embed_count} -> {token_count}")
        model.resize_token_embeddings(token_count)
        embed_count = model.get_input_embeddings().num_embeddings

    special_ids = [int(x) for x in [pad_id, eos_id] if x is not None]
    max_special_id = max(special_ids) if special_ids else -1
    if max_special_id >= embed_count:
        raise ValueError(
            f"PAD/EOS id ngoài embedding vocab: max_special_id={max_special_id}, "
            f"embedding_size={embed_count}. Hãy kiểm tra tokenizer/model hoặc resize embedding."
        )

    model.config.pad_token_id = int(pad_id) if pad_id is not None else None
    model.config.eos_token_id = int(eos_id) if eos_id is not None else None
    return model


def audit_dataset_batch(dataset, data_collator, vocab_size, name):
    if dataset is None or len(dataset) == 0:
        return
    sample_size = min(8, len(dataset))
    batch = [dataset[i] for i in range(sample_size)]
    tensors = data_collator(batch)
    input_ids = tensors["input_ids"]
    labels = tensors["labels"]
    input_min = int(input_ids.min().item())
    input_max = int(input_ids.max().item())
    valid_labels = labels[labels != -100]
    label_min = int(valid_labels.min().item()) if valid_labels.numel() else -100
    label_max = int(valid_labels.max().item()) if valid_labels.numel() else -100
    if input_min < 0 or input_max >= vocab_size or label_max >= vocab_size:
        raise ValueError(
            f"{name} batch có token id ngoài vocab: "
            f"input_range=[{input_min}, {input_max}], "
            f"label_range=[{label_min}, {label_max}], vocab_size={vocab_size}"
        )
    print(
        f"{name} token audit OK | input_range=[{input_min}, {input_max}] "
        f"| label_range=[{label_min}, {label_max}] | vocab_size={vocab_size}"
    )


def print_trainable_parameters(model):
    trainable = 0
    total = 0
    for _, param in model.named_parameters():
        total += param.numel()
        if param.requires_grad:
            trainable += param.numel()
    pct = 100 * trainable / max(1, total)
    print(f"Trainable params: {trainable:,} / {total:,} ({pct:.4f}%)")


tmp_model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
tmp_model = ensure_model_token_embeddings(tmp_model, tokenizer, PAD_ID, EOS_ID)
MODEL_VOCAB_SIZE = tmp_model.get_input_embeddings().num_embeddings
del tmp_model
gc.collect()
torch.cuda.empty_cache()

def stratified_eval_sample_by_type(records, n, seed=SEED, type_key="type"):
    """
    Lấy eval subset theo type:
    - Không lấy 500 dòng đầu để tránh bias theo thứ tự.
    - Nếu n >= số type, đảm bảo mỗi type có ít nhất 1 mẫu.
    - Phần còn lại random từ pool còn lại.
    """
    records = [r for r in records if r.get("response_vi")]

    if not records or n is None or n <= 0:
        return []

    n = min(int(n), len(records))
    rng = random.Random(seed)

    groups = defaultdict(list)
    for r in records:
        rec_type = str(r.get(type_key, "unknown"))
        groups[rec_type].append(r)

    for rec_type in groups:
        rng.shuffle(groups[rec_type])

    selected = []

    # Bảo đảm mỗi type có ít nhất 1 sample nếu đủ budget.
    type_names = sorted(groups.keys())
    if n >= len(type_names):
        for rec_type in type_names:
            selected.append(groups[rec_type].pop())

    remaining_slots = n - len(selected)

    # Fill phần còn lại bằng random từ toàn bộ pool còn lại.
    remaining_pool = []
    for rec_type in type_names:
        remaining_pool.extend(groups[rec_type])

    rng.shuffle(remaining_pool)
    selected.extend(remaining_pool[:remaining_slots])

    rng.shuffle(selected)
    return selected


train_ds = MathDataset(train_records, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH)

eval_records_for_trainer = stratified_eval_sample_by_type(
    valid_records,
    TRAINER_EVAL_SAMPLES,
    seed=SEED,
)

eval_ds = (
    MathDataset(eval_records_for_trainer, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH)
    if eval_records_for_trainer
    else None
)

collator = PadCollator(pad_id=PAD_ID)

print("Eval trainer type distribution:")
print(Counter(r.get("type", "unknown") for r in eval_records_for_trainer))

audit_dataset_batch(train_ds, collator, MODEL_VOCAB_SIZE, "train")
audit_dataset_batch(eval_ds, collator, MODEL_VOCAB_SIZE, "eval")

effective_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count() if CUDA_OK else 0)
steps_per_epoch = math.ceil(len(train_ds) / effective_batch)
total_train_steps = steps_per_epoch * EPOCHS
WARMUP_STEPS = max(1, int(total_train_steps * WARMUP_RATIO)) if total_train_steps > 0 else 0
print("train source:", globals().get("TRAIN_SOURCE", "train_records"))
print("preprocessed train file:", PREPROCESSED_TRAIN_FILE)
print("train samples:", len(train_ds), "| eval samples:", len(eval_ds) if eval_ds else 0)
print("effective batch:", effective_batch)
print("steps/epoch:", steps_per_epoch)
print(f"Total steps for {EPOCHS} epochs:", total_train_steps)
print("warmup steps:", WARMUP_STEPS)


def make_training_args():
    use_bf16 = bool(CUDA_OK and torch.cuda.is_bf16_supported())
    use_fp16 = bool(CUDA_OK and not use_bf16)
    kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        logging_steps=LOGGING_STEPS,
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        dataloader_num_workers=4 if IS_KAGGLE else 0,
        gradient_checkpointing=False,
    )
    sig = inspect.signature(TrainingArguments.__init__)
    has_eval = eval_ds is not None
    
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch" if has_eval else "no"
    else:
        kwargs["evaluation_strategy"] = "epoch" if has_eval else "no"
    if "bf16" in sig.parameters:
        kwargs["bf16"] = use_bf16
    if "fp16" in sig.parameters:
        kwargs["fp16"] = use_fp16
    # V2: dùng adamw_torch_fused trên CUDA hiện đại
    if "optim" in sig.parameters and CUDA_OK:
        kwargs["optim"] = "adamw_torch_fused"
    if not CUDA_OK:
        if "use_cpu" in sig.parameters:
            kwargs["use_cpu"] = True
        elif "no_cuda" in sig.parameters:
            kwargs["no_cuda"] = True
    return TrainingArguments(**kwargs)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resize token embeddings: 50257 -> 50258


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Eval trainer type distribution:
Counter({'GSM_AnsAug': 111, 'GSM_Rephrased': 90, 'MATH_AnsAug': 84, 'GSM_FOBAR': 62, 'MATH_Rephrased': 55, 'GSM_SV': 54, 'MATH_FOBAR': 26, 'MATH_SV': 18})
train token audit OK | input_range=[8, 50257] | label_range=[8, 50257] | vocab_size=50258
eval token audit OK | input_range=[8, 50257] | label_range=[8, 50257] | vocab_size=50258
train source: preprocessed_file:/kaggle/working/data/train_preprocessed.json
preprocessed train file: /kaggle/working/data/train_preprocessed.json
train samples: 95257 | eval samples: 500
effective batch: 64
steps/epoch: 1489
Total steps for 2 epochs: 2978
warmup steps: 59


In [10]:
# 10. Audit target sau preprocessing trước khi train
RESPONSE_AUDIT_PATH = WORK_DIR / "response_audit_report.json"
AUDIT_SAMPLES_PER_TYPE = 20
AUDIT_TOKEN_SAMPLE_SIZE = min(2000, len(train_records))
PLACEHOLDER_RATE_THRESHOLD = 0.02
MIN_CALC_RATE_WARNING = 0.35

PLACEHOLDER_PATTERNS = [
    r"Tính theo dữ kiện trong đề",
    r"Tính theo dữ liệu trong đề",
    r"Dựa vào dữ kiện trong đề",
    r"Lời giải:\s*(?:Tính theo dữ kiện trong đề\.?\s*)?Đáp án là",
]

CALC_PATTERNS = [
    r"\d\s*(?:\+|-|\*|/|=|×|÷)\s*\d",
    r"(?:cộng|trừ|nhân|chia|bằng|tổng|hiệu|tích|thương|suy ra|ta có)",
    r"\$[^$]*(?:\+|-|=|\\frac|\\times|\\div)[^$]*\$",
]

ANSWER_AT_END_RE = re.compile(
    # r"\nĐáp án là\s*[:：]?\s*([^\n]+)\s*$",
    r"(?:^|\n)\s*Đáp án là\s*[:：]?\s*([^\n]+)\s*$",
    flags=re.IGNORECASE
)


def is_placeholder_response(text):
    text = str(text or "")
    return any(re.search(p, text, flags=re.IGNORECASE) for p in PLACEHOLDER_PATTERNS)


def has_intermediate_calculation(text):
    text = str(text or "")
    body = ANSWER_AT_END_RE.sub("", text)
    return any(re.search(p, body, flags=re.IGNORECASE) for p in CALC_PATTERNS)


def final_answer_at_end(rec):
    text = str(rec.get("response_vi", ""))
    m = ANSWER_AT_END_RE.search(text)
    if not m:
        return False
    gold = str(rec.get("answer_text") or "").strip()
    return (not gold) or (gold in m.group(1).strip())


def record_response_audit_row(rec):
    response = str(rec.get("response_vi", ""))
    return {
        "id": rec.get("id"),
        "type": rec.get("type", "unknown"),
        "task_group": get_task_group(rec.get("type", "unknown")),
        "response_words": word_count(response),
        "response_tokens": len(encode_no_special(response)) + 1,
        "was_truncated": bool(rec.get("was_truncated")),
        "placeholder_like": is_placeholder_response(response),
        "has_intermediate_calculation": has_intermediate_calculation(response),
        "final_answer_at_end": final_answer_at_end(rec),
        "answer_text": rec.get("answer_text"),
        "response_preview": response[:500],
    }


audit_rows = [record_response_audit_row(r) for r in train_records]
audit_df = pd.DataFrame(audit_rows)

# Label density audit: tỷ lệ token thực sự được dùng để train (labels != -100)
token_sample_indices = list(range(len(train_records)))
if len(token_sample_indices) > AUDIT_TOKEN_SAMPLE_SIZE:
    token_sample_indices = random.sample(token_sample_indices, AUDIT_TOKEN_SAMPLE_SIZE)

label_rows = []
for idx in tqdm(token_sample_indices, desc="label density audit"):
    item = train_ds[idx]
    total = int(sum(item["attention_mask"]))
    labeled = int(sum(1 for x in item["labels"] if x != -100))
    rec = train_records[idx]
    label_rows.append({
        "id": rec.get("id"),
        "type": rec.get("type", "unknown"),
        "task_group": get_task_group(rec.get("type", "unknown")),
        "labeled_tokens": labeled,
        "total_tokens": total,
        "label_token_ratio": labeled / max(1, total),
    })

label_df = pd.DataFrame(label_rows)
summary_df = (
    audit_df.merge(label_df[["id", "label_token_ratio"]], on="id", how="left")
    .groupby(["type", "task_group"], dropna=False)
    .agg(
        n=("id", "count"),
        response_words_p50=("response_words", "median"),
        response_words_p95=("response_words", lambda s: s.quantile(0.95)),
        response_tokens_p50=("response_tokens", "median"),
        response_tokens_p95=("response_tokens", lambda s: s.quantile(0.95)),
        label_ratio_mean=("label_token_ratio", "mean"),
        label_ratio_p05=("label_token_ratio", lambda s: s.quantile(0.05)),
        placeholder_rate=("placeholder_like", "mean"),
        calc_rate=("has_intermediate_calculation", "mean"),
        truncated_rate=("was_truncated", "mean"),
        final_answer_missing_or_moved_rate=("final_answer_at_end", lambda s: 1 - s.mean()),
    )
    .reset_index()
)

print("Response/target audit summary:")
display(summary_df.round(4))

print("\nMẫu response_vi theo từng type sau preprocessing:")
for t in sorted(audit_df["type"].dropna().unique()):
    print("\n" + "=" * 90)
    print(f"TYPE: {t} | TASK: {get_task_group(t)}")
    type_rows = audit_df[audit_df["type"] == t]
    n_show = AUDIT_SAMPLES_PER_TYPE + (2 if "SV" in t or "FOBAR" in t else 0)
    for _, row in type_rows.head(n_show).iterrows():
        print("-" * 90)
        print(f"id={row['id']} | words={row['response_words']} | placeholder={row['placeholder_like']} | calc={row['has_intermediate_calculation']} | truncated={row['was_truncated']} | final_at_end={row['final_answer_at_end']}")
        print(row["response_preview"])

placeholder_rate = float(audit_df["placeholder_like"].mean()) if len(audit_df) else 1.0
calc_rate = float(audit_df["has_intermediate_calculation"].mean()) if len(audit_df) else 0.0
final_missing_rate = float((~audit_df["final_answer_at_end"]).mean()) if len(audit_df) else 1.0
truncated_rate = float(audit_df["was_truncated"].mean()) if len(audit_df) else 0.0
min_label_ratio = float(label_df["label_token_ratio"].min()) if len(label_df) else 0.0
mean_label_ratio = float(label_df["label_token_ratio"].mean()) if len(label_df) else 0.0

TARGET_READY_FOR_TRAIN = (
    placeholder_rate <= PLACEHOLDER_RATE_THRESHOLD
    and final_missing_rate == 0.0
    and min_label_ratio > 0.0
)

response_audit_report = {
    "summary_by_type": summary_df.to_dict("records"),
    "overall": {
        "n": len(audit_rows),
        "placeholder_rate": placeholder_rate,
        "calc_rate": calc_rate,
        "final_answer_missing_or_moved_rate": final_missing_rate,
        "truncated_rate": truncated_rate,
        "label_ratio_mean": mean_label_ratio,
        "label_ratio_min_sampled": min_label_ratio,
        "target_ready_for_train": TARGET_READY_FOR_TRAIN,
    },
    "sample_rows": audit_rows[:80],
    "label_density_sample": label_rows[:200],
}
save_json(response_audit_report, RESPONSE_AUDIT_PATH)
print("\nWrote:", RESPONSE_AUDIT_PATH)

if placeholder_rate > PLACEHOLDER_RATE_THRESHOLD:
    print(f"WARNING: placeholder_like rate = {placeholder_rate:.2%}. Cần khôi phục/generate lại reasoning target trước khi train.")
if calc_rate < MIN_CALC_RATE_WARNING:
    print(f"WARNING: calc_rate = {calc_rate:.2%}. Response có thể thiếu lời giải trung gian thật.")
if final_missing_rate > 0:
    print(f"WARNING: {final_missing_rate:.2%} response không có final answer ở cuối; kiểm tra smart truncation trước khi train.")
if min_label_ratio <= 0:
    print("WARNING: Có sample trong audit có label_token_ratio = 0; đây là lỗi supervision nghiêm trọng.")

print("TARGET_READY_FOR_TRAIN:", TARGET_READY_FOR_TRAIN)


label density audit:   0%|          | 0/2000 [00:00<?, ?it/s]

Response/target audit summary:


,type,task_group,n,response_words_p50,response_words_p95,response_tokens_p50,response_tokens_p95,label_ratio_mean,label_ratio_p05,placeholder_rate,calc_rate,truncated_rate,final_answer_missing_or_moved_rate
0,GSM_AnsAug,DIRECT_ANSWER,18713,82.0,152.0,98.0,184.00,0.4242,0.2994,0.0,0.9809,0.0015,0.0
1,GSM_FOBAR,REVERSE_PARAM,10016,142.0,225.0,177.0,293.25,0.5103,0.4067,0.0,0.9957,0.0356,0.0
2,GSM_Rephrased,DIRECT_ANSWER,20028,83.0,154.0,99.0,184.65,0.4319,0.3213,0.0,0.9835,0.0008,0.0
3,GSM_SV,SOLVE_FOR_VARIABLE,9823,186.0,257.0,222.0,322.00,0.5665,0.4831,0.0,0.9940,0.0813,0.0
4,MATH_AnsAug,DIRECT_ANSWER,16967,73.0,165.0,134.0,318.00,0.5403,0.3544,0.0,0.9861,0.0321,0.0
5,MATH_FOBAR,REVERSE_PARAM,3665,181.0,277.0,275.0,370.00,0.6037,0.4382,0.0,0.9793,0.3836,0.0
6,MATH_Rephrased,DIRECT_ANSWER,12468,78.0,174.0,135.0,326.65,0.5435,0.3654,0.0,0.9865,0.0337,0.0
7,MATH_SV,SOLVE_FOR_VARIABLE,3577,161.0,275.2,208.0,355.00,0.5304,0.1426,0.0,0.9435,0.1809,0.0



Mẫu response_vi theo từng type sau preprocessing:

TYPE: GSM_AnsAug | TASK: DIRECT_ANSWER
------------------------------------------------------------------------------------------
id=0 | words=85 | placeholder=False | calc=True | truncated=False | final_at_end=True
Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó, tức là 84 * 2/3 = 56 khách. Vậy tổng số khách là 84 + 56 = 140 khách. Người phục vụ luôn làm thêm 10 đĩa nên tổng số đĩa cần dùng là 140 + 10 = 150 đĩa. Mỗi đĩa sẽ có 8 ngọn măng tây trên đó, như vậy tổng số ngọn măng tây cần thiết là 150 * 8 = 1200 ngọn măng tây.
Đáp án là: 1200
------------------------------------------------------------------------------------------
id=3 | words=118 | placeholder=False | calc=True | truncated=False | final_at_end=True
Hans đặt bàn cho 12 người, trong đó có 2 trẻ em nên có 12 - 2 = 10 người lớn. Hans phải trả 3$ cho mỗi người lớn nên tổng chi phí cho người lớn là 10 * 3 = 30$. Anh ta cũng phải trả 1$ cho mỗi đứa trẻ nên tổng c

In [11]:
# 10. Train và lưu checkpoint
if RUN_TRAIN:
    if not globals().get("TARGET_READY_FOR_TRAIN", False):
        raise RuntimeError(
            "TARGET_READY_FOR_TRAIN=False. Hãy xử lý response_vi placeholder/truncation/label audit trước khi train."
        )
    if not CUDA_OK:
        raise RuntimeError(
            "Không có GPU CUDA dùng được cho full training. Hãy đổi Accelerator sang T4/V100/A100."
        )
    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = ensure_model_token_embeddings(model, tokenizer, PAD_ID, EOS_ID)
    # model.gradient_checkpointing_enable()
    model.config.use_cache = False
    
    if USE_LORA:
        if not PEFT_AVAILABLE:
            raise RuntimeError("USE_LORA=True nhưng peft chưa import được. Hãy cài/enable peft trước khi chạy.")
        
        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            target_modules=LORA_TARGET_MODULES,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            task_type="CAUSAL_LM",
            fan_in_fan_out=True,   # quan trọng với GPT-2 Conv1D
        )
        model = get_peft_model(model, lora_config)
        print("Using LoRA config:")
        print(lora_config)
    
    print_trainable_parameters(model)

    trainer = Trainer(
        model=model,
        args=make_training_args(),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
    )

    start = time.time()
    train_output = trainer.train()
    print("Train minutes:", round((time.time() - start) / 60, 2))
    print(train_output)

    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Skip train. Inference sẽ dùng checkpoint nếu có, nếu không dùng base model.")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resize token embeddings: 50257 -> 50258
Using LoRA config:
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path='/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese', revision=None, inference_mode=False, r=32, target_modules={'c_proj', 'c_attn', 'c_fc'}, exclude_modules=None, lora_alpha=64, lora_dropout=0.05, fan_in_fan_out=True, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)
Trainable params: 4,718,59

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.405500,1.395845
2,1.355246,1.363871


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Train minutes: 248.51
TrainOutput(global_step=2978, training_loss=1.4828466962374962, metrics={'train_runtime': 14910.2989, 'train_samples_per_second': 12.777, 'train_steps_per_second': 0.2, 'total_flos': 4.928951736675533e+16, 'train_loss': 1.4828466962374962, 'epoch': 2.0})


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


In [12]:
# 11. Hàm sinh lời giải V2 — batch + beam search + custom StoppingCriteria
MAX_NEW_TOKENS = 256
NUM_BEAMS = 2
DO_SAMPLE = False
REPETITION_PENALTY = 1.3
NO_REPEAT_NGRAM_SIZE = 4
INFER_BATCH_SIZE = 16
EARLY_STOPPING = True

BASELINE_SOURCE = "pure_model"
BASELINE_DECODING = "beam2_antiloop_v4"
# BASELINE_DECODING = "beam" if NUM_BEAMS > 1 else "greedy"


class AnswerStoppingCriteria(StoppingCriteria):
    """
    Dừng generation khi model đã sinh một answer anchor + số đầu tiên.
    Không đợi model tự kết thúc vì GPT-2 hiện dễ sinh đuôi rác sau đáp án.
    """
    def __init__(self, tokenizer, prompt_lens, eos_id, check_every=4):
        super().__init__()
        self.tokenizer = tokenizer
        self.prompt_lens = prompt_lens
        self.eos_id = eos_id
        self.pattern = re.compile(
            r"(?:Đáp\s*án\s*là|Đáp\s*án|Câu\s*trả\s*lời\s*là|Kết\s*quả\s*là|Answer|The answer is)\s*[:：]?\s*[\[\{\(]?\s*[-+]?\d",
            re.IGNORECASE,
        )
        self.check_every = check_every
        self._step = 0

    def __call__(self, input_ids, scores, **kwargs):
        self._step += 1
        if self._step % self.check_every != 0:
            return False

        done = []
        for i in range(input_ids.shape[0]):
            tail = input_ids[i][-120:].tolist()
            text = self.tokenizer.decode(tail, skip_special_tokens=True)
            done.append(bool(self.pattern.search(text)))

        return all(done)
        

def save_json(obj, path):
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def postprocess_output(text):
    """
    Chuẩn hóa output về:
    <reasoning trước anchor>
    Đáp án là: <số đầu tiên sau anchor>

    Mục tiêu: loại bỏ đuôi rác sau đáp án, không để evaluator lấy số cuối sai.
    """
    text = str(text or "").strip()

    # Cắt các marker bắt đầu sample/prompt mới nếu model lặp prompt.
    for marker in [
        "\nCâu hỏi:",
        "\nQuestion:",
        "\nBài toán:",
        "\n[Loại:",
        "\n###",
        "\nGiải bài toán",
        "\nLoại bài:",
        "\n[TASK:",
    ]:
        pos = text.find(marker)
        if pos >= 0:
            text = text[:pos].strip()

    matches = list(ANSWER_ANCHOR_RE.finditer(text))
    if matches:
        m = matches[0]  # với model output: dùng anchor đầu tiên
        reasoning = text[:m.start()].strip()
        tail = text[m.end():]
        ans = first_answer_unit(tail)

        if ans is not None:
            if reasoning:
                return reasoning + f"\nĐáp án là: {ans}"
            return f"Đáp án là: {ans}"

        # Nếu có anchor nhưng chưa đọc được số, chỉ giữ ngắn lại.
        return text[:m.end() + 80].strip()

    return text


def ensure_anchor(text):
    """
    Nếu model không sinh anchor nhưng có số cuối thì gắn anchor.
    Đây chỉ là fallback; evaluation chính vẫn ưu tiên số đầu tiên sau anchor.
    """
    text = str(text or "").strip()

    if ANSWER_ANCHOR_RE.search(text):
        return postprocess_output(text)

    last_num = extract_answer_text(text, allow_last_number=True)
    if last_num:
        return text.rstrip() + f"\nĐáp án là: {last_num}"

    return text
    

def load_model_for_generation(model_dir, tokenizer, dtype, device):
    model_dir = Path(model_dir)
    adapter_config = model_dir / "adapter_config.json"

    if adapter_config.exists():
        if not PEFT_AVAILABLE:
            raise RuntimeError("Checkpoint là LoRA adapter nhưng peft chưa import được.")

        print("Detected LoRA adapter checkpoint:", model_dir)
        base_model = AutoModelForCausalLM.from_pretrained(
            str(MODEL_DIR),
            torch_dtype=dtype,
            local_files_only=True,
        )
        base_model = ensure_model_token_embeddings(base_model, tokenizer, PAD_ID, EOS_ID)
        model = PeftModel.from_pretrained(
            base_model,
            str(model_dir),
            local_files_only=True,
        )
    else:
        print("Detected full model checkpoint:", model_dir)
        model = AutoModelForCausalLM.from_pretrained(
            str(model_dir),
            torch_dtype=dtype,
            local_files_only=True,
        )
        model = ensure_model_token_embeddings(model, tokenizer, PAD_ID, EOS_ID)

    model = model.to(device)
    model.config.use_cache = True
    model.eval()
    return model


def generate_predictions(model_dir, records, output_path, name):
    model_dir = Path(model_dir)
    output_path = Path(output_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if device.type == "cuda" else torch.float32

    gen_tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)

    gen_tokenizer.pad_token_id = PAD_ID
    gen_tokenizer.eos_token_id = EOS_ID
    gen_tokenizer.padding_side = "left"

    if getattr(gen_tokenizer, "pad_token", None) is None:
        if getattr(gen_tokenizer, "eos_token", None) is not None:
            gen_tokenizer.pad_token = gen_tokenizer.eos_token
        else:
            gen_tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

    model = load_model_for_generation(model_dir, gen_tokenizer, dtype, device)

    vocab_size = model.get_input_embeddings().num_embeddings
    outputs = []
    start_all = time.time()

    # Sort records by query length để batch tương đối đồng đều
    order = sorted(range(len(records)), key=lambda i: len(records[i].get("query_vi", "")))
    sorted_records = [records[i] for i in order]

    with torch.inference_mode():
        for batch_start in tqdm(range(0, len(sorted_records), INFER_BATCH_SIZE), desc=name):
            batch = sorted_records[batch_start: batch_start + INFER_BATCH_SIZE]
            prompts = [build_prompt(r) for r in batch]
            enc = gen_tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(device)
            input_ids = enc["input_ids"].clamp(min=0, max=vocab_size - 1)
            attention_mask = enc["attention_mask"]
            prompt_lens = attention_mask.sum(dim=1).tolist()

            stopping = StoppingCriteriaList([
                AnswerStoppingCriteria(gen_tokenizer, prompt_lens, EOS_ID, check_every=4)
            ])

            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                num_beams=NUM_BEAMS,
                early_stopping=EARLY_STOPPING,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=PAD_ID,
                eos_token_id=EOS_ID,
                stopping_criteria=stopping,
            )

            # Với left padding, prompt nằm bên trái, generated = từ input_ids.shape[1] trở đi
            prompt_len_tensor = input_ids.shape[1]
            for i, rec in enumerate(batch):
                new_tokens = gen[i, prompt_len_tensor:]
                text = gen_tokenizer.decode(new_tokens, skip_special_tokens=True)
                text = postprocess_output(text)
                text = ensure_anchor(text)
                rec_type = rec.get("type", "unknown")
                outputs.append({
                    "id": rec.get("id"),
                    "query_vi": rec["query_vi"],
                    "type": rec_type,
                    "task_group": get_task_group(rec_type),
                    "source": BASELINE_SOURCE,
                    "decoding": BASELINE_DECODING,
                    "model_output": text,
                    "_sort_index": batch_start + i,
                })

    # Sắp xếp lại theo thứ tự gốc (map qua order)
    inv_order = {sorted_idx: orig_idx for orig_idx, sorted_idx in enumerate(order)}
    outputs_final = [None] * len(outputs)
    for o in outputs:
        sorted_pos = o.pop("_sort_index")
        original_pos = order[sorted_pos]
        outputs_final[original_pos] = o
    outputs_final = [o for o in outputs_final if o is not None]

    save_json(outputs_final, output_path)
    print("Wrote:", output_path)
    print("Minutes:", round((time.time() - start_all) / 60, 2))

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return outputs_final


In [13]:
# 12. Sinh output validation
RUN_VALIDATION = True
MODEL_FOR_INFERENCE = OUTPUT_DIR if OUTPUT_DIR.exists() else MODEL_DIR

if RUN_VALIDATION and valid_records:
    valid_outputs = generate_predictions(MODEL_FOR_INFERENCE, valid_records, VALID_OUTPUT_PATH, name="validation")
    print("\nOutput mẫu:")
    print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:1200])
else:
    valid_outputs = []
    print("Skip validation generation")


`torch_dtype` is deprecated! Use `dtype` instead!


Detected LoRA adapter checkpoint: /kaggle/working/gpt2_math_baseline_ckpt


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resize token embeddings: 50257 -> 50258


validation:   0%|          | 0/63 [00:00<?, ?it/s]

Wrote: /kaggle/working/valid_output.json
Minutes: 12.85

Output mẫu:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "task_group": "DIRECT_ANSWER",
  "source": "pure_model",
  "decoding": "beam2_antiloop_v4",
  "model_output": "Nếu Susan đang chơi cờ bàn thì cô ấy sẽ di chuyển 2 ô từ ô đầu tiên đến ô cuối cùng của mình. Nếu cô ấy muốn di chuyển thêm 5 ô nữa thì cô ấy phải di chuyển thêm 4 ô nữa. Vì vậy, cô ấy sẽ cần tổng cộng 48 ô. Nếu cô ta muốn di chuyển nhiều hơn 8 ô thì cô ấy cần phải di chuyển 8 ô nữa. Do đó, Susan cần di chuyển tổng cộng 48 + 8 ô = 72 ô.\nĐáp án là: 8"
}


In [14]:
# 13. Đánh giá validation + lưu valid_report.json
CASES_TO_SHOW = 8
VALID_EVAL_ROWS_PATH = WORK_DIR / "valid_eval_rows.jsonl"
VALID_EVAL_ROWS_JSON_PATH = WORK_DIR / "valid_eval_rows.json"


def extract_query_numbers(text):
    nums = []
    for raw in NUM_RE.findall(str(text or "")):
        val = parse_number(raw)
        if val is not None:
            nums.append(val)
    return nums


def close_number(a, b, rel_tol=1e-9, abs_tol=1e-9):
    if a is None or b is None:
        return False
    return abs(a - b) <= max(abs_tol, rel_tol * max(1.0, abs(a), abs(b)))


def detect_repeated_ngram(text, min_n=4, max_n=10, min_repeat=4):
    words = re.findall(r"\S+", str(text or ""))
    if len(words) < min_n * min_repeat:
        return {
            "has_loop": False,
            "loop_repeat_count": 0,
            "loop_ngram": None,
        }

    best_count = 0
    best_ngram = None

    for n in range(min_n, max_n + 1):
        counts = Counter()
        for i in range(0, len(words) - n + 1):
            ng = tuple(words[i:i+n])
            counts[ng] += 1

        if counts:
            ng, cnt = counts.most_common(1)[0]
            if cnt > best_count:
                best_count = cnt
                best_ngram = " ".join(ng)

    return {
        "has_loop": best_count >= min_repeat,
        "loop_repeat_count": int(best_count),
        "loop_ngram": best_ngram,
    }


def arithmetic_eval_binary(a, op, b):
    if op in ["+", "＋"]:
        return a + b
    if op in ["-", "−"]:
        return a - b
    if op in ["*", "×", "x", "X"]:
        return a * b
    if op in ["/", "÷"]:
        if abs(b) < 1e-12:
            return None
        return a / b
    return None


SIMPLE_EQUATION_RE = re.compile(
    r"([-+]?\d+(?:[.,]\d+)?)\s*"
    r"([+\-−*/×xX÷])\s*"
    r"([-+]?\d+(?:[.,]\d+)?)\s*"
    r"=\s*"
    r"([-+]?\d+(?:[.,]\d+)?)"
)


def verify_simple_arithmetic(text, max_checks=20, tol=1e-6):
    """
    Verifier nhẹ: chỉ kiểm tra các phép nhị phân đơn giản a op b = c.
    Không cố giải toàn bộ reasoning.
    """
    text = str(text or "")
    checks = []
    failed = 0
    passed = 0

    for m in SIMPLE_EQUATION_RE.finditer(text):
        if len(checks) >= max_checks:
            break

        a = parse_number(m.group(1))
        op = m.group(2)
        b = parse_number(m.group(3))
        c = parse_number(m.group(4))

        if a is None or b is None or c is None:
            continue

        expected = arithmetic_eval_binary(a, op, b)
        if expected is None:
            continue

        ok = abs(expected - c) <= max(tol, tol * max(1.0, abs(expected), abs(c)))

        checks.append({
            "expr": m.group(0),
            "expected": expected,
            "actual": c,
            "ok": bool(ok),
        })

        if ok:
            passed += 1
        else:
            failed += 1

    if not checks:
        status = "unknown"
    elif failed > 0:
        status = "fail"
    else:
        status = "pass"

    return {
        "arithmetic_status": status,
        "arithmetic_checked": len(checks),
        "arithmetic_passed": passed,
        "arithmetic_failed": failed,
        "arithmetic_fail_examples": [x for x in checks if not x["ok"]][:5],
    }


def build_verifier_report_for_output(model_output, pred_answer, pred_num):
    text = str(model_output or "")

    loop_info = detect_repeated_ngram(text)
    arith_info = verify_simple_arithmetic(text)

    return {
        "answer_anchor_found": bool(ANSWER_ANCHOR_RE.search(text)),
        "pred_answer_exists": pred_answer is not None,
        "pred_num_exists": pred_num is not None,
        **loop_info,
        **arith_info,
    }
    

def classify_error(score, pred_answer, pred_num, gold_num, query_nums, rec_type, verifier=None):
    verifier = verifier or {}
    if pred_answer is None or pred_num is None:
        return "parse_or_no_number"
    if score == 10:
        return "correct_1pct"
    if score in (5, 1):
        return "near_miss_numeric"
    if verifier.get("has_loop"):
        return "loop"
    if verifier.get("arithmetic_status") == "fail":
        return "arithmetic_inconsistent"
    if any(close_number(pred_num, q) for q in query_nums):
        return "copy_input_number"
    if "SV" in str(rec_type):
        return "wrong_solve_for_variable"
    if "FOBAR" in str(rec_type):
        return "wrong_reverse_param"
    return "wrong_numeric_answer"


def align_by_id(preds, golds):
    if all("id" in x for x in preds) and all("id" in x for x in golds):
        pred_map = {str(x["id"]): x for x in preds}
        return [(pred_map[str(g["id"])], g) for g in golds if str(g["id"]) in pred_map]
    return list(zip(preds, golds))


def evaluate_predictions(preds, golds):
    rows = []
    for row_index, (pred, gold) in enumerate(align_by_id(preds, golds)):
        gold_answer = extract_answer_text(
            gold.get("response_vi"),
            allow_last_number=True,
            prefer_first_anchor=False,
        )
        
        pred_answer = extract_answer_text(
            pred.get("model_output"),
            allow_last_number=False,
            prefer_first_anchor=True,
        )
        
        # fallback defensive: nếu vì lý do nào đó output không có anchor
        if pred_answer is None:
            pred_answer = extract_answer_text(
                pred.get("model_output"),
                allow_last_number=True,
                prefer_first_anchor=True,
            )
        gold_num = parse_number(gold_answer)
        pred_num = parse_number(pred_answer)
        rel_err = relative_error(pred_num, gold_num)
        score = score_one(rel_err, pred_answer is not None)
        rec_type = gold.get("type")
        query_nums = extract_query_numbers(gold.get("query_vi"))
        
        verifier = build_verifier_report_for_output(
            pred.get("model_output"),
            pred_answer,
            pred_num,
        )
        rows.append({
            "row_index": row_index,
            "id": gold.get("id"),
            "type": rec_type,
            "task_group": get_task_group(rec_type),
            "source": pred.get("source", "pure_model"),
            "decoding": pred.get("decoding", BASELINE_DECODING if "BASELINE_DECODING" in globals() else None),
            "query_vi": gold.get("query_vi"),
            "model_output": pred.get("model_output"),
            "gold_answer": gold_answer,
            "pred_answer": pred_answer,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "query_numbers": query_nums,
            "rel_error": rel_err,
            "extractable": pred_answer is not None,
            "score": score,
            "answer_anchor_found": verifier["answer_anchor_found"],
            "has_loop": verifier["has_loop"],
            "loop_repeat_count": verifier["loop_repeat_count"],
            "loop_ngram": verifier["loop_ngram"],
            "arithmetic_status": verifier["arithmetic_status"],
            "arithmetic_checked": verifier["arithmetic_checked"],
            "arithmetic_passed": verifier["arithmetic_passed"],
            "arithmetic_failed": verifier["arithmetic_failed"],
            "arithmetic_fail_examples": verifier["arithmetic_fail_examples"],
            "error_bucket": classify_error(score, pred_answer, pred_num, gold_num, query_nums, rec_type, verifier),
        })
    return rows


def score_summary(rows):
    n = len(rows)
    raw = sum(r["score"] for r in rows)
    return {
        "n": n,
        "raw_score": raw,
        "max_raw_score": 10 * n,
        "score_10": raw / n if n else 0,
        "extractable_rate": sum(r["extractable"] for r in rows) / n if n else 0,
        "buckets": {str(s): sum(r["score"] == s for r in rows) for s in [10, 5, 1, 0]},
    }


def show_cases(title, rows):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    if not rows:
        print("Không có case")
        return
    cols = ["row_index", "id", "type", "score", "rel_error", "gold_answer", "pred_answer", "query_vi", "model_output"]
    display(pd.DataFrame(rows[:CASES_TO_SHOW])[cols])


if valid_outputs:
    eval_rows = evaluate_predictions(valid_outputs, valid_records)
    summary = score_summary(eval_rows)
    print("Validation summary:")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    eval_df = pd.DataFrame(eval_rows)
    by_type = eval_df.groupby("type").apply(lambda x: pd.Series(score_summary(x.to_dict("records")))).reset_index()
    display(by_type[["type", "n", "score_10", "extractable_rate", "raw_score", "max_raw_score", "buckets"]])

    print("\nError buckets by type:")
    error_by_type = (
        eval_df.groupby(["type", "task_group", "error_bucket"])
        .size()
        .reset_index(name="count")
        .sort_values(["type", "count"], ascending=[True, False])
    )
    display(error_by_type)

    save_records_jsonl(eval_rows, VALID_EVAL_ROWS_PATH)
    save_json(eval_rows, VALID_EVAL_ROWS_JSON_PATH)
    print("Wrote:", VALID_EVAL_ROWS_PATH)
    print("Wrote:", VALID_EVAL_ROWS_JSON_PATH)

    verifier_summary = {
        "anchor_found_rate": float(eval_df["answer_anchor_found"].mean()) if len(eval_df) else 0.0,
        "loop_rate": float(eval_df["has_loop"].mean()) if len(eval_df) else 0.0,
        "arithmetic_checked_rate": float((eval_df["arithmetic_checked"] > 0).mean()) if len(eval_df) else 0.0,
        "arithmetic_fail_rate_all": float((eval_df["arithmetic_status"] == "fail").mean()) if len(eval_df) else 0.0,
        "arithmetic_fail_rate_checked": (
            float(
                ((eval_df["arithmetic_status"] == "fail") & (eval_df["arithmetic_checked"] > 0)).sum()
                / max(1, (eval_df["arithmetic_checked"] > 0).sum())
            )
            if len(eval_df)
            else 0.0
        ),
    }
    
    print("\nVerifier summary:")
    print(json.dumps(verifier_summary, ensure_ascii=False, indent=2))
    
    verifier_by_type = (
        eval_df.groupby(["type", "task_group"])
        .agg(
            n=("type", "size"),
            loop_rate=("has_loop", "mean"),
            anchor_found_rate=("answer_anchor_found", "mean"),
            arithmetic_checked_rate=("arithmetic_checked", lambda x: float((x > 0).mean())),
            arithmetic_fail_rate=("arithmetic_status", lambda x: float((x == "fail").mean())),
        )
        .reset_index()
    )
    
    display(verifier_by_type)

    show_cases("Một vài case đúng (score=10)", [r for r in eval_rows if r["score"] == 10])
    show_cases("Một vài case sai (score=0)", [r for r in eval_rows if r["score"] == 0 and r["extractable"]])
    show_cases("Một vài case không tách được đáp án", [r for r in eval_rows if not r["extractable"]])

    # Lưu valid_report.json
    report = {
        "summary": summary,
        "by_type": by_type.to_dict("records"),
        "verifier_summary": verifier_summary,
        "verifier_by_type": verifier_by_type.to_dict("records"),
        "config": {
            "epochs": EPOCHS,
            "lr": LEARNING_RATE,
            "effective_batch": effective_batch,
            "max_length": MAX_LENGTH,
            "max_new_tokens": MAX_NEW_TOKENS,
            "num_beams": NUM_BEAMS,
            "do_sample": DO_SAMPLE,
            "source": BASELINE_SOURCE if "BASELINE_SOURCE" in globals() else "pure_model",
            "decoding": BASELINE_DECODING if "BASELINE_DECODING" in globals() else None,
            "prompt_template": PROMPT_TEMPLATE,
            "instruction": INSTRUCTION,
            "task_group_map": TASK_GROUP_MAP,
            "type_label_map": TYPE_LABEL_MAP,
            "experiment_name": EXPERIMENT_NAME,
            "use_lora": USE_LORA,
            "lora_r": LORA_R if USE_LORA else None,
            "lora_alpha": LORA_ALPHA if USE_LORA else None,
            "lora_dropout": LORA_DROPOUT if USE_LORA else None,
            "lora_target_modules": LORA_TARGET_MODULES if USE_LORA else None,
            "run_mode": "pure_model_no_retrieval_no_rule_no_voting",
        },
        "eval_rows_path": str(VALID_EVAL_ROWS_PATH),
        "eval_rows_json_path": str(VALID_EVAL_ROWS_JSON_PATH),
        "error_buckets_by_type": error_by_type.to_dict("records"),
        "wrong_cases_preview": [r for r in eval_rows if r["score"] == 0][:30],
    }
    save_json(report, VALID_REPORT_PATH)
    print(f"\nWrote: {VALID_REPORT_PATH}")
else:
    eval_rows = []
    summary = None
    print("Không có validation output để đánh giá")


Validation summary:
{
  "n": 1000,
  "raw_score": 748,
  "max_raw_score": 10000,
  "score_10": 0.748,
  "extractable_rate": 0.999,
  "buckets": {
    "10": 49,
    "5": 19,
    "1": 163,
    "0": 769
  }
}


/tmp/ipykernel_23/3483646478.py:271: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_type = eval_df.groupby("type").apply(lambda x: pd.Series(score_summary(x.to_dict("records")))).reset_index()


,type,n,score_10,extractable_rate,raw_score,max_raw_score,buckets
0,GSM_AnsAug,209,0.555024,1.00000,116,2090,"{'10': 5, '5': 6, '1': 36, '0': 162}"
1,GSM_FOBAR,122,1.040984,1.00000,127,1220,"{'10': 10, '5': 0, '1': 27, '0': 85}"
2,GSM_Rephrased,197,0.527919,1.00000,104,1970,"{'10': 4, '5': 6, '1': 34, '0': 153}"
3,GSM_SV,97,0.639175,1.00000,62,970,"{'10': 4, '5': 1, '1': 17, '0': 75}"
4,MATH_AnsAug,173,0.959538,0.99422,166,1730,"{'10': 13, '5': 3, '1': 21, '0': 136}"
5,MATH_FOBAR,45,1.044444,1.00000,47,450,"{'10': 4, '5': 0, '1': 7, '0': 34}"
6,MATH_Rephrased,116,0.698276,1.00000,81,1160,"{'10': 6, '5': 1, '1': 16, '0': 93}"
7,MATH_SV,41,1.097561,1.00000,45,410,"{'10': 3, '5': 2, '1': 5, '0': 31}"



Error buckets by type:


,type,task_group,error_bucket,count
0,GSM_AnsAug,DIRECT_ANSWER,arithmetic_inconsistent,110
3,GSM_AnsAug,DIRECT_ANSWER,near_miss_numeric,42
5,GSM_AnsAug,DIRECT_ANSWER,wrong_numeric_answer,41
1,GSM_AnsAug,DIRECT_ANSWER,copy_input_number,9
2,GSM_AnsAug,DIRECT_ANSWER,correct_1pct,5
4,GSM_AnsAug,DIRECT_ANSWER,parse_or_no_number,2
6,GSM_FOBAR,REVERSE_PARAM,arithmetic_inconsistent,46
10,GSM_FOBAR,REVERSE_PARAM,wrong_reverse_param,38
9,GSM_FOBAR,REVERSE_PARAM,near_miss_numeric,27
8,GSM_FOBAR,REVERSE_PARAM,correct_1pct,10


...


Một vài case sai (score=0)


,row_index,id,type,score,rel_error,gold_answer,pred_answer,query_vi,model_output
0,0,0,GSM_Rephrased,0,0.783784,37,8,"Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nh...",Nếu Susan đang chơi cờ bàn thì cô ấy sẽ di chuyển 2 ô từ ô đầu tiên đến ô cuối cùng của mình. Nếu cô ấy muốn di chuyển thêm 5 ô nữa thì cô ấy phải di chuyển thêm 4 ô nữa. Vì vậ...
1,1,1,MATH_Rephrased,0,0.789474,19,4,"Nếu $\angle PQR = \angle PRQ$, và độ dài của QR và PR lần lượt là 5 và 7 thì chu vi của tam giác PQR là bao nhiêu?","Chúng ta có thể viết lại phương trình thành $\angle PRQ = 5$ và $\angle PR Q = 7$. Để tìm độ dài của một tam giác, chúng ta có thể nhân chéo: $\angle PRq = 5$ hoặc $\angle PR q..."
2,2,2,MATH_SV,0,0.750000,8,2,Một con súc sắc tám mặt có các mặt được đánh số từ 1 đến X. Giá trị kỳ vọng của con xúc xắc là 4.5. Giá trị của biến X chưa biết là bao nhiêu?,"Để giải quyết vấn đề này, chúng ta cần xác định giá trị của x, đại diện cho giá trị kỳ vọng mà con xúc xắc có thể đạt được. Hãy chia nhỏ thông tin đã cho: Số mặt được đánh dấu:..."
3,3,3,GSM_Rephrased,0,0.948718,39,2,"Sau khi bắt đầu với 50 quả bóng bay, Claire đưa cho một bé gái 1 quả bóng bay, mất 12 quả bóng bay, cho thêm 9 quả bóng nữa và lấy 11 quả bóng bay từ đồng nghiệp của mình. Hiện...",Filton đưa cho bé gái 1/2*1 = 2 quả bóng bay. Cô ấy đưa cho cô ấy 1/3*1 = 3 quả bóng bay nên cô ấy đưa cho 3 * 2 = 4 quả bóng bay . Cô ấy lấy tổng cộng 2 + 4 = 6 quả bóng bay t...
4,5,5,GSM_AnsAug,0,0.994444,90,60/120,"Hans đặt phòng ở khách sạn. Khách sạn có 10 tầng, mỗi tầng có 10 phòng giống nhau. Do xảy ra tai nạn nên tầng cuối cùng không còn chỗ cho khách. Xem xét không có khách nào khác...","Tầng 1 có 10 phòng. Tầng 2 có 10 phòng nên tầng 1 có 10 * 10 = 30 phòng. Tầng 3 có 10 phòng, mỗi tầng đều có 10 phòng như vậy. Tầng 4 có 10 phòng và mỗi tầng đều 10 phòng như t..."
5,6,6,GSM_SV,0,1.500000,12,30,Bà Dunbar đang cắm hoa cho đám cưới của cháu gái bà. Cô ấy cần làm 5 bó hoa và 7 món đồ trang trí bàn ăn. Cô sử dụng x bông hồng trắng để trang trí mỗi bàn và 5 bông hồng trắng...,"Để giải quyết vấn đề này, chúng ta cần xác định giá trị của x, đại diện cho số bông hồng trắng mà bà Dunbar cần làm mỗi bàn. Hãy chia nhỏ thông tin đã cho: Số bông hồng trắng: ..."
6,9,9,GSM_AnsAug,0,11.000000,5,60,"Bob được hỗ trợ tiền thuê nhà vì anh ấy có thu nhập thấp. Nếu anh ta được tăng lương 0.50 USD/giờ và làm việc 40 giờ một tuần, anh ta sẽ thực sự kiếm được bao nhiêu tiền một tu...","Bob có thu nhập hàng tháng là $0.50 x 0.50 = $60. Anh ấy đã làm việc tổng cộng 40 giờ mỗi tuần, vì vậy anh ấy sẽ làm việc 40 x 40 = $60 mỗi tuần. Vì vậy, anh ấy sẽ kiếm được $6..."
7,10,10,GSM_FOBAR,0,1.000000,1,2,John phải thay vòng bi cho những chiếc máy mà anh ấy làm việc cùng. Anh ta có 10 chiếc máy và mỗi chiếc có 30 vòng bi. Thông thường nó có giá x $ cho mỗi ổ bi nhưng hiện tại đa...,"John có 10 máy, mỗi máy có 30 vòng quay nên anh ấy có tổng cộng 10 * 30 = 60 vòng quay. Anh ấy cũng có 10 máy nên anh ấy còn lại 60 - 10 = 20 vòng quay. Tổng số vòng quay là tổ..."



Một vài case không tách được đáp án


,row_index,id,type,score,rel_error,gold_answer,pred_answer,query_vi,model_output
0,280,280,MATH_AnsAug,0,None,28,None,"Gọi GCF(a, b) là chữ viết tắt của ước chung lớn nhất của a và b, và LCM(c, d) là chữ viết tắt của bội số chung nhỏ nhất của c và d. GCF(LCM(8, 14), LCM(7, 12)) là gì?",Chúng ta có thể viết lại biểu thức này thành biểu thức riêng của mình. Biểu thức này có thể được viết bằng cách sử dụng các ký hiệu khác nhau. Biểu thức đầu tiên là biểu thức t...



Wrote: /kaggle/working/valid_report.json


In [15]:
# 14. Sinh test_predictions.json cho Phase 2
RUN_TEST_INFERENCE = True

if RUN_TEST_INFERENCE and test_records:
    test_outputs = generate_predictions(MODEL_FOR_INFERENCE, test_records, TEST_OUTPUT_PATH, name="test")
    # Loại bỏ field _sort_index nếu còn (defensive)
    test_outputs_clean = [
        {k: v for k, v in o.items() if not k.startswith("_")}
        for o in test_outputs
    ]
    save_json(test_outputs_clean, TEST_OUTPUT_PATH)
    print("\nTest output mẫu:")
    print(json.dumps(test_outputs_clean[:2], ensure_ascii=False, indent=2)[:1200])
else:
    print("Không có test.json, bỏ qua bước test inference")


Không có test.json, bỏ qua bước test inference


In [16]:
# 15. Kiểm tra file đầu ra
for p in [
    WORK_DIR,
    OUTPUT_DIR,
    VALID_OUTPUT_PATH,
    VALID_EVAL_ROWS_PATH,
    VALID_REPORT_PATH,
    TEST_OUTPUT_PATH,
]:
    p = Path(p)
    if p.exists():
        size = p.stat().st_size if p.is_file() else "<dir>"
        print(p, "|", size)

print("\nDone.")


/kaggle/working | <dir>
/kaggle/working/gpt2_math_baseline_ckpt | <dir>
/kaggle/working/valid_output.json | 956838
/kaggle/working/valid_eval_rows.jsonl | 1485120
/kaggle/working/valid_report.json | 64448

Done.
